# Toy charging-station example (presentation)

Edit the **station** and **EV list** below, run the episode, then use the metric tables and figures in slides.

Arrivals are **exact** (not Poisson). The helpers live under `toy_demo/` and call the existing env, metrics, and plot modules.

In [ ]:
from pathlib import Path
import sys

# Repo root = parent of toy_demo/ (works even if the kernel cwd is toy_demo/)
ROOT = Path.cwd()
if ROOT.name == "toy_demo":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt

from policy.queue.fifo import FIFOQueuePolicy
from policy.queue.closest_power_match import ClosestPowerMatchQueuePolicy
from metrics.validate import validate_episode, report_queueing_laws
from visualization.pile_power import plot_pile_nozzle_power
from visualization.ev_curves import plot_ev_theory_vs_sim

from toy_demo.scenario import ToyEVSpec, ToyStationSpec, default_four_ev_specs
from toy_demo.runner import run_toy_episode, metrics_table, finished_ev_table

print("ROOT =", ROOT)

## 1. Station layout

Change pile count, nozzles per pile, bricks, and brick power (kW).

In [ ]:
station = ToyStationSpec(
    n_piles=2,
    n_nozzles=2,
    n_bricks=5,
    p_brick=25.0,
    queue_capacity=10,
)
station

## 2. EVs and arrival times

Each row is one EV: `arrival_time` (min), `battery_kwh`, `s_i`, `s_f`.
Start from the default four-EV cast, or replace the list entirely.

In [ ]:
ev_specs = default_four_ev_specs()

# --- edit freely, e.g. ---
# ev_specs = [
#     ToyEVSpec(id=0, arrival_time=0.0, battery_kwh=50.0, s_i=0.20, s_f=0.80),
#     ToyEVSpec(id=1, arrival_time=3.0, battery_kwh=100.0, s_i=0.15, s_f=0.85),
#     ToyEVSpec(id=2, arrival_time=6.0, battery_kwh=50.0, s_i=0.25, s_f=0.75),
#     ToyEVSpec(id=3, arrival_time=10.0, battery_kwh=150.0, s_i=0.10, s_f=0.80),
# ]

for s in ev_specs:
    print(
        f"EV {s.id}: arr={s.arrival_time:.1f} min, "
        f"C_b={s.battery_kwh:.0f} kWh, s_i={s.s_i:.2f}, s_f={s.s_f:.2f}"
    )

## 3. Policy and horizon

`horizon` is when the clock stops (`SIM_OVER`). If anyone is still charging at the end, increase it.

In [ ]:
policy = FIFOQueuePolicy()
# policy = ClosestPowerMatchQueuePolicy()

horizon = 120.0  # minutes; or None → last arrival + 90

env = run_toy_episode(
    station,
    ev_specs,
    policy=policy,
    horizon=horizon,
    seed=0,
    policy_seed=1,
)
print(
    f"Done at t={env.engine.current_time:.2f} min | "
    f"arrived={len(env.engine.metrics.arrived_evs)}, "
    f"finished={len(env.engine.metrics.finished_evs)}, "
    f"dropped={len(env.engine.metrics.dropped_evs)}"
)

## 4. Metrics

In [ ]:
summary = metrics_table(env)
display(summary)

per_ev = finished_ev_table(env)
display(per_ev)

validate_episode(env, verbose=True, raise_on_fail=False)
report_queueing_laws(env, verbose=True)

## 5. Visuals

Pile power vs time (one figure per pile), then theory-vs-sim curves for selected EVs.

In [ ]:
t_end = float(env.engine.current_time)
pile_figs = []
for pile_id in range(station.n_piles):
    fig = plot_pile_nozzle_power(
        env,
        pile_id=pile_id,
        t_start=0.0,
        t_end=t_end,
        show_theory=True,
        show_sim_bms=True,
        show_actual=True,
        show_charge_change_lines=True,
        show_charge_change_vlines=True,
        show_event_lines=True,
        show_trace_labels=False,
    )
    pile_figs.append(fig)
    display(fig)

In [ ]:
ev_ids = [s.id for s in ev_specs]
ev_figs = plot_ev_theory_vs_sim(
    env,
    ev_ids=ev_ids,
    charts=["P-S", "T-S", "P-T"],
    show_theory=True,
    show_sim_bms=True,
    show_sim_actual=True,
    show_recorded_samples=True,
)
for fig in ev_figs:
    display(fig)

## 6. Optional: save figures for the deck

Uncomment to write PNGs under `toy_demo/output/`.

In [ ]:
# out = ROOT / "toy_demo" / "output"
# out.mkdir(parents=True, exist_ok=True)
# for i, fig in enumerate(pile_figs):
#     fig.savefig(out / f"pile_{i}.png", dpi=200, bbox_inches="tight")
# for ev_id, fig in zip(ev_ids, ev_figs):
#     fig.savefig(out / f"ev_{ev_id}.png", dpi=200, bbox_inches="tight")
# summary.to_csv(out / "summary_metrics.csv")
# per_ev.to_csv(out / "finished_evs.csv", index=False)
# print("Saved to", out)